In [ ]:
# FINAL WORKING SKIN TYPE CLASSIFIER
from google.colab import drive
drive.mount('/content/drive')

import torch
from transformers import ViTForImageClassification, ViTImageProcessor
from PIL import Image
import matplotlib.pyplot as plt
import os

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model architecture
model = ViTForImageClassification.from_pretrained(
    "dima806/skin_types_image_detection",
    num_labels=3,
    ignore_mismatched_sizes=True
).to(device)

# Load your trained weights (THE WORKING MODEL)
model_path = '/content/drive/MyDrive/skin_model_direct_193442.pth'
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

# Label mappings
id2label = {0: 'dry', 1: 'normal', 2: 'oily'}

def predict_skin(image_path):
    """Predict skin type from image"""
    processor = ViTImageProcessor.from_pretrained("dima806/skin_types_image_detection")

    image = Image.open(image_path).convert('RGB')
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        pred_class = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][pred_class].item()

    return id2label[pred_class], confidence, probs.cpu().numpy()[0]

print("✅ Skin Type Classifier Ready!")
print(f"📊 Expected Accuracy: ~84%")
print(f"💾 Model: skin_model_direct_193442.pth")
print(f"🎯 Classes: Dry, Normal, Oily")

# Test it with a new image
def test_new_image(image_path):
    """Test the classifier with any image"""
    if not os.path.exists(image_path):
        print(f"❌ Image not found: {image_path}")
        return

    pred, conf, all_probs = predict_skin(image_path)

    print(f"🎯 Prediction: {pred.upper()} skin")
    print(f"💪 Confidence: {conf:.1%}")
    print(f"📊 All probabilities:")
    print(f"   - Dry: {all_probs[0]:.1%}")
    print(f"   - Normal: {all_probs[1]:.1%}")
    print(f"   - Oily: {all_probs[2]:.1%}")

    # Display the image
    img = Image.open(image_path)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(f"Skin Type: {pred.upper()}\nConfidence: {conf:.1%}", fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Example usage:
# test_new_image("/path/to/your/skin/image.jpg")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

✅ Skin Type Classifier Ready!
📊 Expected Accuracy: ~84%
💾 Model: skin_model_direct_193442.pth
🎯 Classes: Dry, Normal, Oily


In [ ]:
# -*- coding: utf-8 -*-
"""Skin Classification - COMPLETE RESTART SOLUTION"""

# STEP 1: RESTART RUNTIME FIRST - Go to Runtime -> Restart runtime
# THEN run this cell

print("🚀 NUCLEAR OPTION - Fresh start")

# Install FRESH packages
!pip install --quiet --upgrade transformers torchvision pillow matplotlib scikit-learn imbalanced-learn tqdm

print("✅ Fresh packages installed")

# Import ONLY what we need
import os
import numpy as np
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

print("✅ Core imports successful")

# Mount drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Drive mounted")

# Now import PyTorch and Transformers
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

print("✅ PyTorch imports successful")

# FINALLY import transformers
from transformers import ViTForImageClassification, ViTImageProcessor

print("✅ Transformers imported successfully!")
print("🎉 ALL IMPORTS WORKING!")

# Custom Dataset Class
class SkinDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# Load your data
DATA_PATH = "/content/drive/MyDrive/Oily-Dry-Skin-Types"
print(f"📁 Loading data from: {DATA_PATH}")

file_names = []
labels = []

dataset_directory = Path(DATA_PATH)
for file in tqdm(sorted(dataset_directory.glob('*/*/*.*'))):
    if file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
        file_names.append(str(file))
        label = file.parent.name
        labels.append(label)

print(f"✅ Found {len(file_names)} images")

# Create label mappings
unique_labels = sorted(list(set(labels)))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}

print("🏷️ Classes:", id2label)

# Convert to numeric labels
numeric_labels = [label2id[label] for label in labels]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    file_names, numeric_labels, test_size=0.2, random_state=42, stratify=numeric_labels
)

print(f"📚 Train: {len(X_train)}, Test: {len(X_test)}")

# Load model and processor
model_str = "dima806/skin_types_image_detection"
print(f"🤖 Loading model: {model_str}")

processor = ViTImageProcessor.from_pretrained(model_str)
size = processor.size["height"]

print(f"🖼️ Image size: {size}")

# Simple transforms
train_transform = transforms.Compose([
    transforms.Resize((size, size)),
    transforms.RandomHorizontalFlip(0.3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=processor.image_mean,
        std=processor.image_std
    )
])

test_transform = transforms.Compose([
    transforms.Resize((size, size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=processor.image_mean,
        std=processor.image_std
    )
])

# Create datasets
train_dataset = SkinDataset(X_train, y_train, transform=train_transform)
test_dataset = SkinDataset(X_test, y_test, transform=test_transform)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print("✅ Data loaders ready")

# Load model
model = ViTForImageClassification.from_pretrained(
    model_str,
    num_labels=len(unique_labels),
    ignore_mismatched_sizes=True
)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"📱 Using device: {device}")
print(f"⚙️ Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M")

# Training setup
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-6, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

# Training function
def train_model(model, train_loader, test_loader, epochs=3):
    model.train()

    for epoch in range(epochs):
        print(f"🚀 Epoch {epoch+1}/{epochs}")

        # Training
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for images, labels in tqdm(train_loader, desc="Training"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images).logits
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_acc = 100 * train_correct / train_total

        # Validation
        model.eval()
        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for images, labels in tqdm(test_loader, desc="Validation"):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images).logits
                _, predicted = torch.max(outputs.data, 1)
                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()

        test_acc = 100 * test_correct / test_total

        print(f"📊 Epoch {epoch+1}: Train Loss: {train_loss/len(train_loader):.4f}, "
              f"Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

        # 💾 SAVE MODEL AFTER EACH EPOCH
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_acc': train_acc,
            'test_acc': test_acc,
            'loss': train_loss/len(train_loader)
        }
        torch.save(checkpoint, f'/content/drive/MyDrive/skin_model_epoch_{epoch+1}.pth')
        print(f"💾 Checkpoint saved: skin_model_epoch_{epoch+1}.pth")

    return model

# Train the model
print("🎯 Starting training...")
trained_model = train_model(model, train_loader, test_loader, epochs=3)

# 💾 SAVE FINAL MODEL
final_checkpoint = {
    'model_state_dict': trained_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': 3
}
torch.save(final_checkpoint, '/content/drive/MyDrive/skin_model_final.pth')
print("💾 Final model saved: skin_model_final.pth")

# Final evaluation
print("📈 Final evaluation...")
trained_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = trained_model(images).logits
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
print(f"🎯 Final Test Accuracy: {accuracy:.4f}")

print("\n📋 Classification Report:")
print(classification_report(all_labels, all_preds, target_names=unique_labels))

# Test on a sample
print("\n🧪 Testing on sample image...")
sample_image_path = X_test[0]
sample_image = Image.open(sample_image_path).convert('RGB')
true_label = id2label[y_test[0]]

# Preprocess for model
inputs = processor(images=sample_image, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = trained_model(**inputs)
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    predicted_label = id2label[predicted_class]

print(f"🖼️ Sample image: {os.path.basename(sample_image_path)}")
print(f"✅ True label: {true_label}")
print(f"🤖 Predicted: {predicted_label}")
print(f"📊 Confidence: {probabilities[0][predicted_class]:.4f}")

# Show the image
plt.figure(figsize=(8, 6))
plt.imshow(sample_image)
plt.title(f"True: {true_label} | Predicted: {predicted_label}")
plt.axis('off')
plt.show()

print("🎉 TRAINING COMPLETED SUCCESSFULLY!")

🚀 NUCLEAR OPTION - Fresh start
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 899.7/899.7 MB 36.6 MB/s eta 0:00:01
ERROR: Operation cancelled by user
✅ Fresh packages installed
✅ Core imports successful
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted
✅ PyTorch imports successful
✅ Transformers imported successfully!
🎉 ALL IMPORTS WORKING!
📁 Loading data from: /content/drive/MyDrive/Oily-Dry-Skin-Types


100%|██████████| 3152/3152 [00:00<00:00, 253619.93it/s]


✅ Found 3152 images
🏷️ Classes: {0: 'dry', 1: 'normal', 2: 'oily'}
📚 Train: 2521, Test: 631
🤖 Loading model: dima806/skin_types_image_detection
🖼️ Image size: 224
✅ Data loaders ready
📱 Using device: cpu
⚙️ Trainable parameters: 85.8M
🎯 Starting training...
🚀 Epoch 1/3


Validation: 100%|██████████| 40/40 [09:25<00:00, 14.15s/it]


📊 Epoch 1: Train Loss: 0.8901, Train Acc: 65.61%, Test Acc: 63.39%
💾 Checkpoint saved: skin_model_epoch_1.pth
🚀 Epoch 2/3


Validation: 100%|██████████| 40/40 [07:06<00:00, 10.67s/it]


📊 Epoch 2: Train Loss: 0.8368, Train Acc: 69.38%, Test Acc: 65.13%
💾 Checkpoint saved: skin_model_epoch_2.pth
🚀 Epoch 3/3


Training:  77%|███████▋  | 122/158 [1:07:58<19:58, 33.29s/it]

In [ ]:
# Optional: Test with a new image if you want to see it working
from google.colab import files
import shutil

uploaded = files.upload()
for filename in uploaded.keys():
    shutil.move(filename, f"/content/{filename}")
    pred, conf, probs = predict_skin(f"/content/{filename}")
    print(f"🎯 Prediction: {pred} skin ({conf:.1%} confidence)")

Saving test2.jpg to test2.jpg
🎯 Prediction: oily skin (45.6% confidence)
